In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import normflows as nf
import time
from torch.distributions import Normal

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class NBase(nn.Module):
    def __init__(self, dim, init_loc=0.0, init_log_scale=-2.5):
        super().__init__()
        self.dim = dim
        self.loc = nn.Parameter(torch.full((dim,), float(init_loc)))
        self.raw_log_scale = nn.Parameter(torch.full((dim,), float(init_log_scale)))

    def log_scale(self):
        return torch.clamp(self.raw_log_scale, min=-5.0, max=2.0)

    def scale(self):
        return torch.exp(self.log_scale())

    def rsample(self, num_samples):
        eta = torch.randn(
            num_samples,
            self.dim,
            device=self.loc.device,
            dtype=self.loc.dtype,
        )
        z0 = self.loc.unsqueeze(0) + self.scale().unsqueeze(0) * eta
        return eta, z0

    def log_prob(self, z0):
        log_scale = self.log_scale().unsqueeze(0)
        var = torch.exp(2.0 * log_scale)
        return -0.5 * (
            ((z0 - self.loc.unsqueeze(0)) ** 2) / var
            + 2.0 * log_scale
            + math.log(2.0 * math.pi)
        ).sum(dim=1)

In [4]:
def simfun1(n=180, p=100, seed=123, snr=3.0, true_prop=0.1, device=None, dtype=torch.float32,):

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    X = rng.standard_normal((n, p)).astype(np.float32)
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)

    n_active = int(p * true_prop)
    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))

    beta_true = np.zeros(p, dtype=np.float32)
    magnitudes = rng.uniform(0.3, 2.0, size=n_active).astype(np.float32)
    signs = rng.choice([-1.0, 1.0], size=n_active).astype(np.float32)
    beta_true[active_idx] = signs * magnitudes

    signal = X @ beta_true
    sigma2 = np.var(signal) / snr
    sigma = np.sqrt(sigma2)

    y = signal + sigma * rng.standard_normal(n).astype(np.float32)
    y = y - y.mean()

    X_t = torch.tensor(X, dtype=dtype, device=device)
    y_t = torch.tensor(y, dtype=dtype, device=device)
    beta_true_t = torch.tensor(beta_true, dtype=dtype, device=device)

    info = {"n": n, "p": p, "n_active": n_active, "sigma2": float(sigma2), "sigma": float(sigma), "active_idx": active_idx, "snr": snr,}

    return X_t, y_t, beta_true_t, info

In [5]:
class AffineCoupling(nn.Module):
    def __init__(self, dim, mask, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        self.dim = dim
        self.register_buffer("mask", mask.view(1, dim))
        self.scale_clip = float(scale_clip)

        widths = [dim] + [hidden_units] * num_hidden_layers + [dim]
        self.s_net = nf.nets.MLP(widths, init_zeros=True)
        self.t_net = nf.nets.MLP(widths, init_zeros=True)

    def forward(self, x, return_logdet=False):
        x_masked = x * self.mask

        raw_log_scale = self.s_net(x_masked) * (1.0 - self.mask)
        shift = self.t_net(x_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        y = x_masked + (1.0 - self.mask) * (x * torch.exp(log_scale) + shift)

        if return_logdet:
            logdet = log_scale.sum(dim=-1)
            return y, logdet
        return y

    def inverse(self, y, return_logdet=False):
        y_masked = y * self.mask

        raw_log_scale = self.s_net(y_masked) * (1.0 - self.mask)
        shift = self.t_net(y_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        x = y_masked + (1.0 - self.mask) * ((y - shift) * torch.exp(-log_scale))

        if return_logdet:
            logdet = (-log_scale).sum(dim=-1)
            return x, logdet
        return x


class FlowMap(nn.Module):
    def __init__(self, dim, K=4, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        self.dim = dim

        base_mask = torch.tensor(
            [1.0 if i % 2 == 0 else 0.0 for i in range(dim)],
            dtype=torch.float32,
        )

        layers = []
        for k in range(K):
            mask = base_mask if (k % 2 == 0) else (1.0 - base_mask)
            layers.append(
                AffineCoupling(
                    dim=dim,
                    mask=mask,
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                    scale_clip=scale_clip,
                )
            )

        self.layers = nn.ModuleList(layers)

    def forward(self, x, return_logdet=False):
        z = x
        if return_logdet:
            total_logdet = x.new_zeros(x.shape[0])

        for layer in self.layers:
            if return_logdet:
                z, logdet = layer(z, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                z = layer(z)

        if return_logdet:
            return z, total_logdet
        return z

    def inverse(self, z, return_logdet=False):
        x = z
        if return_logdet:
            total_logdet = z.new_zeros(z.shape[0])

        for layer in reversed(self.layers):
            if return_logdet:
                x, logdet = layer.inverse(x, return_logdet=True)
                total_logdet = total_logdet + logdet
            else:
                x = layer.inverse(x)

        if return_logdet:
            return x, total_logdet
        return x

In [6]:
class Relaxedsas(nn.Module):
    def __init__(self, X, y, sigma2, tau, g_theta):
        super().__init__()
        if g_theta is None:
            raise ValueError("g_theta must be provided.")

        self.register_buffer("X", X)
        self.register_buffer("y", y)
        self.register_buffer("sigma2", torch.tensor(float(sigma2), dtype=X.dtype, device=X.device))
        self.register_buffer("tau", torch.tensor(float(tau), dtype=X.dtype, device=X.device))

        self.n, self.p = X.shape
        self.dim = 2 * self.p + 1
        self.g_theta = g_theta

    def set_tau(self, tau):
        self.tau.fill_(float(tau))

    def decode(self, eps):
        xi = self.g_theta(eps)

        p = self.p
        s = xi[:, :p]
        u = xi[:, p:2 * p]
        t = xi[:, 2 * p:2 * p + 1]

        gate = torch.sigmoid((u - t) / self.tau)
        beta = s * gate

        return {"eps": eps, "xi": xi, "s": s, "u": u, "t": t, "gate": gate, "beta": beta,}

    def log_joint(self, eps):
        dec = self.decode(eps)
        beta = dec["beta"]

        fit = self.X @ beta.T
        resid = self.y[:, None] - fit

        loglik = -0.5 * (
            resid.pow(2).sum(dim=0) / self.sigma2
            + self.n * torch.log(2.0 * torch.pi * self.sigma2)
        )

        log_p0_eps = -0.5 * (
            eps.pow(2) + math.log(2.0 * math.pi)
        ).sum(dim=1)

        return loglik + log_p0_eps

In [7]:
class FlowVI(nn.Module):
    def __init__(self, q0, posterior_flow, generative_model):
        super().__init__()
        self.q0 = q0
        self.posterior_flow = posterior_flow
        self.generative_model = generative_model

    def sample_posterior(self, num_samples):
        _, z0 = self.q0.rsample(num_samples)
        eps, logdet = self.posterior_flow(z0, return_logdet=True)
        log_q_eps = self.q0.log_prob(z0) - logdet
        return eps, log_q_eps

    def neg_elbo(self, num_samples=256, elbo_beta=1.0):
        eps, log_q_eps = self.sample_posterior(num_samples)
        log_joint = self.generative_model.log_joint(eps)
        return (log_q_eps - elbo_beta * log_joint).mean()


def build_flow_vi(X, y, sigma2, tau, K_q=8, K_g=8, hidden_units=64, num_hidden_layers=2,):
    dim = 2 * X.shape[1] + 1

    g_theta = FlowMap(
        dim=dim,
        K=K_g,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=2.0,
    )

    generative_model = Relaxedsas(X=X, y=y, sigma2=sigma2, tau=tau, g_theta=g_theta,)

    q0 = NBase(dim=dim, init_loc=0.0, init_log_scale=-2.5,)
    
    posterior_flow = FlowMap(
        dim=dim,
        K=K_q,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        scale_clip=2.0,
    )

    return FlowVI(q0=q0, posterior_flow=posterior_flow, generative_model=generative_model,)

In [21]:
def train_flow(model, epochs=1000, num_samples=128, lr=5e-5, tau_start=1.0, tau_end=0.05,
    anneal_ratio=0.7, grad_clip=3.0, print_every=100, elbo_beta=1.0, pip_R=2000,):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    tau_hist = []
    grad_hist = []
    
    anneal_epochs = max(1, int(round(epochs * anneal_ratio)))

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        if epoch <= anneal_epochs:
            frac = (epoch - 1) / max(anneal_epochs - 1, 1)
            tau_now = tau_start * (tau_end / tau_start) ** frac
        else:
            tau_now = float(tau_end)

        model.generative_model.set_tau(tau_now)
        tau_hist.append(float(tau_now))

        loss = model.neg_elbo(num_samples=num_samples, elbo_beta=elbo_beta)
        loss.backward()

        if grad_clip is not None:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            grad_norm = float(grad_norm)
        else:
            grad_norm = 0.0

        optimizer.step()

        losses.append(float(loss.item()))
        grad_hist.append(grad_norm)
        
        draws = sample_posterior_latents(model, R=pip_R)
        u = draws["u"]
        t = draws["t"]
        if t.ndim == 1:
            t = t.unsqueeze(1)

        pip = (u > t).float().mean(dim=0)
        expected_support = float(pip.sum().item())
        suspect_num = int(((pip > 0.3) & (pip < 0.7)).sum().item())


        if epoch % print_every == 0 or epoch == 1:
            print(
                f"[epoch {epoch:04d}] "
                f"loss={loss.item():.6f} "
                f"tau={tau_now:.6f} "
                f"grad_norm={grad_norm:.6f} "
                f"expected_select= {expected_support}"
                f"suspect_number= {suspect_num}"
            )

    return losses, tau_hist, grad_hist

In [12]:
@torch.no_grad()
def sample_posterior_latents(model, R=2000):
    model.eval()

    q0 = model.q0
    _, z0 = q0.rsample(R)               # [R, d]
    eps, _ = model.posterior_flow(z0, return_logdet=True)
    dec = model.generative_model.decode(eps)

    out = {
        "s": dec["s"].detach().cpu(),          # [R, p]
        "u": dec["u"].detach().cpu(),          # [R, p]
        "t": dec["t"].detach().cpu(),          # [R, 1] or [R]
        "beta": dec["beta"].detach().cpu(),    # [R, p]
    }
    return out

@torch.no_grad()
def pip_summary(model, beta_true=None, R=2000, c=0.5):
    draws = sample_posterior_latents(model, R=R)

    s = draws["s"]
    u = draws["u"]                      # [R, p]
    t = draws["t"]
    beta = draws["beta"]

    if t.ndim == 1:
        t = t.unsqueeze(1)              # [R, 1]

    ind = (u > t).int()                 # [R, p]
    pip = ind.float().mean(dim=0)       # [p]
    beta_soft_mean = beta.mean(dim=0)        # [p]
    beta_hard_mean = (s * ind.float()).mean(dim=0)

    support_c = (pip > c).int().numpy()

    df = pd.DataFrame({
    "j": np.arange(pip.numel()),
    "pip": pip.numpy(),
    "beta_hard_mean": beta_hard_mean.numpy(),
    "beta_soft_mean": beta_soft_mean.numpy(),
    "selected_c": support_c,
})

    metrics = {
        "R": R,
        "pip_threshold": c,
        "expected_support_size": float(pip.sum().item()),
        "n_selected_at_c": int(support_c.sum()),
        "selected_idx_at_c": np.where(support_c == 1)[0].tolist(),
    }

    if beta_true is not None:
        beta_true_cpu = beta_true.detach().cpu()
        truth = (beta_true_cpu.abs() > 1e-12).int().numpy()
        pred = support_c

        tp = int(((pred == 1) & (truth == 1)).sum())
        fp = int(((pred == 1) & (truth == 0)).sum())
        fn = int(((pred == 0) & (truth == 1)).sum())
        tn = int(((pred == 0) & (truth == 0)).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        df["beta_true"] = beta_true_cpu.numpy()
        df["truth"] = truth
        df["pred_c"] = pred

        metrics.update({
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    return df, metrics, draws

In [18]:
def simflow(
    seed=123,
    device=None,
    dtype=torch.float32,
    hidden_units=64,
    num_hidden_layers=2,
    anneal_ratio=1.0,
    n=180,
    p=100,
    snr=3.0,
    true_prop=0.1,
    tau_end=0.5,
    K_q=8,
    K_g=8,
    epochs=1000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=100,
    pip_R=2000,
    pip_c=0.5,
    return_point_diagnostics=True,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X, y, beta_true, sim_info = simfun1(
        n=n,
        p=p,
        seed=seed,
        snr=snr,
        true_prop=true_prop,
        device=device,
        dtype=dtype,
    )

    print("===== Simulation info =====")
    print(sim_info)

    model = build_flow_vi(
        X=X,
        y=y,
        sigma2=sim_info["sigma2"],
        tau=tau_end,
        K_q=K_q,
        K_g=K_g,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
    ).to(device)

    losses, tau_hist, grad_hist = train_flow(
        model=model,
        epochs=epochs,
        num_samples=num_samples,
        lr=lr,
        tau_start=1.0,
        tau_end=tau_end,
        anneal_ratio=anneal_ratio,
        grad_clip=grad_clip,
        print_every=print_every,
        elbo_beta=1.0,
        pip_R=pip_R
    )

    pip_df, pip_metrics, posterior_draws = pip_summary(
        model,
        beta_true=beta_true,
        R=pip_R,
        c=pip_c,
    )

    out = {
        "beta_true": beta_true,
        "sim_info": sim_info,
        "model": model,
        "losses": losses,
        "tau_hist": tau_hist,
        "grad_hist": grad_hist,
        "pip_df": pip_df,
        "pip_metrics": pip_metrics,
        "posterior_draws": posterior_draws,
    }

    return out

def show_simflow_results(out, top_k=20, sort_by="pip"):
    sim_info = out["sim_info"]
    pip_df = out["pip_df"].copy()
    pip_metrics = out["pip_metrics"]

    print("===== Simulation info =====")
    print({
        "n": sim_info["n"],
        "p": sim_info["p"],
        "n_active": sim_info["n_active"],
        "snr": sim_info["snr"],
        "active_idx": sim_info["active_idx"].tolist() if hasattr(sim_info["active_idx"], "tolist") else sim_info["active_idx"],
    })

    print("\n===== Variational PIP summary =====")
    print({
        "R": pip_metrics["R"],
        "pip_threshold": pip_metrics["pip_threshold"],
        "expected_support_size": pip_metrics["expected_support_size"],
        "n_selected_at_c": pip_metrics["n_selected_at_c"],
        "selected_idx_at_c": pip_metrics["selected_idx_at_c"],
    })

    if "precision" in pip_metrics:
        print("\n===== Selection metrics at threshold c =====")
        print({
            "tp": pip_metrics["tp"],
            "fp": pip_metrics["fp"],
            "fn": pip_metrics["fn"],
            "tn": pip_metrics["tn"],
            "precision": pip_metrics["precision"],
            "recall": pip_metrics["recall"],
            "f1": pip_metrics["f1"],
        })

    true_active = set(sim_info["active_idx"].tolist()) if hasattr(sim_info["active_idx"], "tolist") else set(sim_info["active_idx"])
    pip_df["is_true_active"] = pip_df["j"].isin(true_active)

    if sort_by not in pip_df.columns:
        raise ValueError(f"sort_by='{sort_by}' not found in pip_df columns: {list(pip_df.columns)}")

    view_cols = [c for c in [
        "j", "pip", "pip_uncertainty", "beta_hard_mean", "is_true_active", "beta_true"
    ] if c in pip_df.columns]

    top_df = pip_df.sort_values(sort_by, ascending=False).head(top_k)[view_cols]

    print(f"\n===== Top {top_k} variables by {sort_by} =====")
    print(top_df.to_string(index=False))

    return top_df

In [19]:
out_flow1 = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1, pip_R=100, pip_c=0.9,
    tau_end=0.05, anneal_ratio=0.7, K_q=16, K_g=16, hidden_units=64, num_hidden_layers=2,
    epochs=6000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=100,
)

top_df1 = show_simflow_results(out_flow1, top_k=20, sort_by="pip")

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=891.764465 tau=1.000000 grad_norm=275.459290 mc_expected_support= 45.93000030517578mc_ambiguous= 100
[epoch 0100] loss=654.191589 tau=0.931806 grad_norm=188.832077 mc_expected_support= 87.88999938964844mc_ambiguous= 6
[epoch 0200] loss=378.745422 tau=0.867643 grad_norm=116.077530 mc_expected_support= 18.829999923706055mc_ambiguous= 16
[epoch 0300] loss=288.885162 tau=0.807899 grad_norm=116.819054 mc_expected_support= 12.710000038146973mc_ambiguous= 3
[epoch 0400] loss=262.777405 tau=0.752268 grad_norm=73.838242 mc_expected_support= 9.230000495910645mc_ambiguous= 2
[epoch 0500] loss=253.832382 tau=0.700468 grad_norm=78.608788 mc_expected_support= 8.09000015258789mc_ambiguous= 0
[epoch 0600] loss=249.951294 tau=0.652235 grad_norm=80.767784 mc_expected_support= 7.9

In [22]:
out_flow2 = simflow(seed=123, n=120, p=100, snr=2.5, true_prop=0.1, pip_R=100, pip_c=0.9,
    tau_end=0.05, anneal_ratio=1, K_q=16, K_g=16, hidden_units=64, num_hidden_layers=2,
    epochs=6000, num_samples=128, lr=5e-5, grad_clip=3.0, print_every=100,
)

top_df2 = show_simflow_results(out_flow2, top_k=20, sort_by="pip")

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 6.834272766113282, 'sigma': 2.614244205523516, 'active_idx': array([14, 23, 24, 28, 29, 45, 69, 74, 84, 90], dtype=int64), 'snr': 2.5}
[epoch 0001] loss=891.764465 tau=1.000000 grad_norm=275.459290 expected_select= 45.93000030517578suspect_number= 100
[epoch 0100] loss=654.215271 tau=0.951764 grad_norm=188.912216 expected_select= 87.8800048828125suspect_number= 7
[epoch 0200] loss=378.606750 tau=0.905403 grad_norm=116.416542 expected_select= 18.670001983642578suspect_number= 16
[epoch 0300] loss=288.756958 tau=0.861300 grad_norm=107.831612 expected_select= 11.680000305175781suspect_number= 6
[epoch 0400] loss=262.748901 tau=0.819346 grad_norm=72.142227 expected_select= 8.320000648498535suspect_number= 0
[epoch 0500] loss=253.806839 tau=0.779435 grad_norm=81.545258 expected_select= 7.420000076293945suspect_number= 1
[epoch 0600] loss=249.853561 tau=0.741468 grad_norm=74.167755 expected_select= 7.310000419616699su